# Carga MySQL desde Jupyter (Docker)

Ejecutar **dentro del contenedor** `nudat-jupyter` (o con `MYSQL_HOST=127.0.0.1` desde el host si el puerto 3306 está publicado).

Red Docker: el hostname del servicio MySQL es **`mysql`** (no `localhost`).

Prerrequisito: `data/processed/*.csv` generados por `02_clean_etl.ipynb`.

In [ ]:
import os
from sqlalchemy import text
from src.load_db import get_engine, database_url, load_all

print("DATABASE URL (password hidden):")
print(database_url().replace(os.getenv("MYSQL_PASSWORD", "nudat"), "***"))
print("MYSQL_HOST =", os.getenv("MYSQL_HOST", "127.0.0.1"))

In [ ]:
engine = get_engine()
with engine.connect() as conn:
    row = conn.execute(text("SELECT DATABASE(), USER()")).fetchone()
    print("Connected:", row)
    tables = conn.execute(text("SHOW TABLES")).fetchall()
    print("Tables:", [t[0] for t in tables])

In [ ]:
# Carga reproducible desde data/processed/
load_all()

In [ ]:
# Smoke query: N/Z medio por modo (estados base)
q = text("""
SELECT
  ns.dominant_mode,
  COUNT(*) AS n,
  ROUND(AVG(n.N / NULLIF(n.Z, 0)), 3) AS mean_N_over_Z
FROM nuclear_state ns
JOIN nuclide n ON n.nuclide_id = ns.nuclide_id
WHERE ns.level_index = 0
GROUP BY ns.dominant_mode
ORDER BY n DESC
""")
import pandas as pd
pd.read_sql(q, engine)

## DBeaver

- Host: `localhost` (puerto `3306` publicado)
- Database: `nudat`
- User / password: `nudat` / `nudat`
- Luego: click derecho en la BD → **View diagram** / ER diagram para evidenciar el modelo.